# 01 — Data Exploration

Load `train.csv`, split into **labeled** (first 5 955 rows) and **unlabeled** (remaining rows),
visualise the class distribution, and display one sample image per class.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

DATA_DIR = os.path.join("..", "Data", "DFUC2021_train")
CSV_PATH = os.path.join(DATA_DIR, "train.csv")
IMG_DIR  = os.path.join(DATA_DIR, "images")

## 1 — Load and split the CSV

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f"Total rows : {len(df)}")
print(f"Columns    : {list(df.columns)}")
df.head()

In [ ]:
N_LABELED = 5955
LABEL_COLS = ["none", "infection", "ischaemia", "both"]

labeled_df   = df.iloc[:N_LABELED].copy()
unlabeled_df = df.iloc[N_LABELED:].copy()

print(f"Labeled   : {len(labeled_df)}")
print(f"Unlabeled : {len(unlabeled_df)}")

## 2 — Derive a single `label` column from the one-hot encoding

In [ ]:
labeled_df["label"] = labeled_df[LABEL_COLS].values.argmax(axis=1)
labeled_df["label_name"] = labeled_df["label"].map(
    {i: name for i, name in enumerate(LABEL_COLS)}
)
labeled_df[["image", "label", "label_name"]].head(10)

## 3 — Class distribution (bar chart)

In [ ]:
counts = labeled_df["label_name"].value_counts().reindex(LABEL_COLS)

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(counts.index, counts.values, color=["#5b9bd5", "#ed7d31", "#a5a5a5", "#ffc000"])
ax.set_ylabel("Count")
ax.set_title("Class Distribution (labeled subset, n=5 955)")
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 40,
            str(val), ha="center", va="bottom", fontweight="bold")
plt.tight_layout()
plt.show()

## 4 — One sample image per class

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, (cls_idx, cls_name) in zip(axes, enumerate(LABEL_COLS)):
    row = labeled_df[labeled_df["label"] == cls_idx].iloc[0]
    img = Image.open(os.path.join(IMG_DIR, row["image"]))
    ax.imshow(img)
    ax.set_title(f"{cls_name}  (idx {cls_idx})")
    ax.axis("off")

plt.suptitle("One sample per class", fontsize=14)
plt.tight_layout()
plt.show()

## 5 — Quick sanity check on unlabeled subset

In [ ]:
print("Unlabeled rows — label columns are all NaN:")
unlabeled_df[LABEL_COLS].isna().all()